# 🏥 PreventIQ — ML Risk Intelligence Pipeline
### Trains model + precomputes full JSON output for every state & district
---
Run all cells top-to-bottom. At the end, `preventiq_cache.json`, `models/best_model.joblib`, and `models/scaler.joblib` will be saved — hand these to your backend developer.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, json, math, datetime
from pathlib import Path
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, classification_report, confusion_matrix)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from scipy.stats import ttest_ind, pointbiserialr
import joblib

# ── XGBoost (install if missing) ──────────────────────────────────────────
try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
    print("✅ XGBoost available")
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'xgboost', '-q'])
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
    print("✅ XGBoost installed")

print("✅ All imports done")


In [ ]:
df = pd.read_excel("final_health_data.xlsx")
print(f"Dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"States : {df['state_or_ut'].nunique()}")
print(f"Districts: {df['district_name'].nunique()}")
print(f"Target distribution:\n{df['ill'].value_counts()}")
df.head()


In [ ]:
# ── Scale numerical features ─────────────────────────────────────────────
COLS_TO_SCALE = [
    'age', 'household_size', 'chronic_disease', 'hypertension', 'diabetes', 'asthma',
    'missed_visits_12m', 'missed_screenings_12m', 'days_since_last_visit',
    'followup_delay_days', 'screening_completed', 'vaccination_up_to_date',
    'facility_access_km', 'region_health_index'
]
COLS_TO_SCALE = [c for c in COLS_TO_SCALE if c in df.columns]

scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[COLS_TO_SCALE] = scaler.fit_transform(df[COLS_TO_SCALE])

# ── One-hot encode categoricals ───────────────────────────────────────────
CAT_FEATURES = ['age_group', 'locality_type', 'literacy_status', 'income_bracket', 'occupation_type']
CAT_FEATURES = [c for c in CAT_FEATURES if c in df.columns]
df_scaled = pd.get_dummies(df_scaled, columns=CAT_FEATURES, drop_first=False)

print(f"After encoding: {df_scaled.shape}")
print(f"Scaled {len(COLS_TO_SCALE)} numerical + encoded {len(CAT_FEATURES)} categorical features")


In [ ]:
# ── T-test feature selection ─────────────────────────────────────────────
target      = df_scaled['ill']
all_numeric = [c for c in df_scaled.select_dtypes(include=[np.number]).columns if c != 'ill']

t_records = []
for feat in all_numeric:
    ill_g = df_scaled[target == 1][feat].dropna()
    not_g = df_scaled[target == 0][feat].dropna()
    t, p  = ttest_ind(ill_g, not_g, equal_var=False)
    pool  = math.sqrt((ill_g.std()**2 + not_g.std()**2) / 2)
    cd    = (ill_g.mean() - not_g.mean()) / pool if pool > 0 else 0
    t_records.append({'feature': feat, 'p_value': p, 'cohens_d': cd,
                      'mean_ill': ill_g.mean(), 'mean_not': not_g.mean()})

t_df = pd.DataFrame(t_records).sort_values('p_value')

# Keep significant features; strip state-specific binary columns (data leakage)
sig_raw = t_df[t_df['p_value'] < 0.05]['feature'].tolist()
state_leakage = [f for f in sig_raw
                 if f.startswith('is_') and f not in ('is_City','is_Town','is_Village')]
FEATURES = [f for f in sig_raw if f not in state_leakage]

print(f"Significant features: {len(sig_raw)}  →  after removing {len(state_leakage)} state indicators: {len(FEATURES)}")
print("Selected features:", FEATURES)

# Store Cohen's d & p-value per feature for later use
FEATURE_STATS = {row['feature']: {'cohens_d': row['cohens_d'],
                                   'p_value': row['p_value'],
                                   'mean_ill': row['mean_ill'],
                                   'mean_not': row['mean_not']}
                 for _, row in t_df.iterrows() if row['feature'] in FEATURES}


In [ ]:
X = df_scaled[FEATURES].fillna(df_scaled[FEATURES].mean())
y = df_scaled['ill']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
SPW = neg / pos   # scale_pos_weight for XGBoost

print(f"Train: {X_train.shape[0]:,}  Test: {X_test.shape[0]:,}  Features: {X_train.shape[1]}")
print(f"Class balance → not-ill: {neg}, ill: {pos},  scale_pos_weight: {SPW:.3f}")


In [ ]:
MODEL_DIR = Path('models')
MODEL_DIR.mkdir(exist_ok=True)

CANDIDATES = {
    'Logistic Regression':  LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'Decision Tree':        DecisionTreeClassifier(random_state=42, max_depth=10),
    'Random Forest':        RandomForestClassifier(n_estimators=200, random_state=42,
                                                    max_depth=10, class_weight='balanced', n_jobs=-1),
    'Gradient Boosting':    GradientBoostingClassifier(n_estimators=200, random_state=42, max_depth=5),
    'XGBoost':              XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                                          subsample=0.8, colsample_bytree=0.8,
                                          scale_pos_weight=SPW, eval_metric='logloss',
                                          random_state=42, n_jobs=-1, verbosity=0),
    'Naive Bayes':          GaussianNB(),
}

results, trained = [], {}
print("="*65)
for name, clf in CANDIDATES.items():
    clf.fit(X_train, y_train)
    trained[name] = clf
    yp   = clf.predict(X_test)
    prob = clf.predict_proba(X_test)[:, 1]
    row  = {'Model': name,
            'Accuracy':  round(accuracy_score(y_test, yp), 4),
            'Precision': round(precision_score(y_test, yp, zero_division=0), 4),
            'Recall':    round(recall_score(y_test, yp, zero_division=0), 4),
            'F1':        round(f1_score(y_test, yp, zero_division=0), 4),
            'ROC-AUC':   round(roc_auc_score(y_test, prob), 4)}
    results.append(row)
    print(f"{name:<25} Acc={row['Accuracy']} F1={row['F1']} AUC={row['ROC-AUC']}")

results_df = pd.DataFrame(results).sort_values('F1', ascending=False).reset_index(drop=True)
print("\n🏆 Leaderboard:")
print(results_df.to_string(index=False))


In [ ]:
BEST_NAME  = results_df.iloc[0]['Model']
BEST_MODEL = trained[BEST_NAME]
BEST_META  = results_df.iloc[0].to_dict()

print(f"\n🏆 Best Model: {BEST_NAME}")
print(f"   F1={BEST_META['F1']}  AUC={BEST_META['ROC-AUC']}  Accuracy={BEST_META['Accuracy']}")

# Cross-validation
cv    = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_f1 = cross_val_score(BEST_MODEL, X, y, cv=cv, scoring='f1', n_jobs=-1)
print(f"   5-fold CV F1: {cv_f1.mean():.4f} ± {cv_f1.std():.4f}")

# Save model + scaler
joblib.dump(BEST_MODEL, MODEL_DIR / 'best_model.joblib')
joblib.dump(scaler,     MODEL_DIR / 'scaler.joblib')
print(f"\n✅ Saved: models/best_model.joblib, models/scaler.joblib")


In [ ]:
# Get feature importances from best model
if hasattr(BEST_MODEL, 'feature_importances_'):
    importances = dict(zip(FEATURES, BEST_MODEL.feature_importances_))
elif hasattr(BEST_MODEL, 'coef_'):
    importances = dict(zip(FEATURES, np.abs(BEST_MODEL.coef_[0])))
    # normalise to sum to 1
    total = sum(importances.values())
    importances = {k: v/total for k,v in importances.items()}
else:
    importances = {f: 1/len(FEATURES) for f in FEATURES}

# Sort
IMPORTANCES_SORTED = sorted(importances.items(), key=lambda x: x[1], reverse=True)

print("Top 10 feature importances:")
for feat, imp in IMPORTANCES_SORTED[:10]:
    cd = FEATURE_STATS.get(feat, {}).get('cohens_d', 0)
    print(f"  {feat:<30} importance={imp:.4f}  cohen_d={cd:.3f}")

# National feature means (for deviation calculation in drivers)
NATIONAL_MEANS = df_scaled[FEATURES].mean().to_dict()


In [ ]:
# ═══════════════════════════════════════════════════════════════
# HELPER FUNCTIONS — compute each JSON block from a data subset
# ═══════════════════════════════════════════════════════════════

FRIENDLY_NAMES = {
    'missed_screenings_12m':  'Missed Screening Intervals',
    'missed_visits_12m':      'Missed Healthcare Visits',
    'obesity':                'Obesity Prevalence',
    'pgi':                    'Public Health Index (Low)',
    'chronic_disease':        'Chronic Disease Burden',
    'hypertension':           'Hypertension Rate',
    'diabetes':               'Diabetes Prevalence',
    'asthma':                 'Asthma Prevalence',
    'days_since_last_visit':  'Days Since Last Visit',
    'followup_delay_days':    'Follow-up Delay (Days)',
    'region_health_index':    'Regional Health Index',
    'facility_access_km':     'Distance to Facility (km)',
    'age':                    'Elderly Population Cluster',
    'household_size':         'Large Household Size',
    'screening_completed':    'Screening Completion',
    'vaccination_up_to_date': 'Vaccination Coverage',
    'is_Village':             'Rural Village Population',
    'is_Town':                'Semi-urban Town Population',
    'is_City':                'Urban City Concentration',
}

WEATHER_BY_MONTH = {
    1:  ("Winter (dry, cold)", 0.95, "Cold wave possible — elderly risk elevated"),
    2:  ("Winter (dry)", 0.96, "Low weather impact expected"),
    3:  ("Pre-summer (dry heat)", 0.90, "Rising temperatures — outdoor screening difficult"),
    4:  ("Pre-monsoon (hot)", 0.82, "Heat stress risk — hydration support needed"),
    5:  ("Pre-monsoon (high rain probability)", 0.78, "Heavy rainfall expected — mobile clinic effectiveness reduced"),
    6:  ("Monsoon onset", 0.72, "Monsoon active — vector-borne disease risk elevated"),
    7:  ("Peak monsoon", 0.68, "Peak monsoon — field operations severely limited"),
    8:  ("Monsoon (heavy)", 0.70, "Flooding risk — clinic access impacted in low-lying areas"),
    9:  ("Retreating monsoon", 0.80, "Post-flood disease surge risk — dengue alert"),
    10: ("Post-monsoon", 0.88, "Dengue & malaria season — extra vigilance needed"),
    11: ("Early winter", 0.93, "Respiratory illness risk rising"),
    12: ("Winter", 0.94, "Cold-related illness risk — elderly vulnerability high"),
}

def sf(v, d=0.0):
    try:
        x = float(v)
        return d if (math.isnan(x) or math.isinf(x)) else round(x, 4)
    except: return d

def compute_ml_risk(scaled_sub):
    X_sub = scaled_sub[FEATURES].fillna(scaled_sub[FEATURES].mean())
    probs = BEST_MODEL.predict_proba(X_sub)[:, 1]
    avg   = sf(probs.mean())
    lo    = round(float(np.percentile(probs, 10)) * 100, 1)
    hi    = round(float(np.percentile(probs, 90)) * 100, 1)
    pct   = round(avg * 100, 1)
    std   = sf(probs.std())
    level = 'High' if avg > 0.70 else ('Medium' if avg > 0.50 else 'Low')
    conf  = 'High' if std < 0.15 else 'Medium'
    return {
        'current_risk_pct':    pct,
        'risk_level':          level,
        'confidence':          conf,
        'prediction_interval': [lo, hi],
        'model_accuracy':      round(BEST_META['Accuracy'] * 100, 1),
        'model_name':          BEST_NAME,
        'last_trained':        datetime.date.today().isoformat(),
    }

def compute_drivers(scaled_sub):
    local_means = scaled_sub[FEATURES].mean().to_dict()
    drivers = []
    for feat, imp in IMPORTANCES_SORTED:
        local_m  = sf(local_means.get(feat, 0))
        nat_m    = sf(NATIONAL_MEANS.get(feat, 0))
        delta    = local_m - nat_m
        stats    = FEATURE_STATS.get(feat, {})
        cd       = sf(stats.get('cohens_d', 0))
        pv       = sf(stats.get('p_value', 1))
        contrib  = round(imp * abs(delta) * 100, 2)
        direction = 'risk-increasing' if delta > 0 else 'risk-decreasing'
        drivers.append({
            'feature':             feat,
            'label':               FRIENDLY_NAMES.get(feat, feat.replace('_',' ').title()),
            'importance':          round(imp * 100, 2),
            'direction':           direction,
            'cohens_d':            round(cd, 3),
            'p_value':             round(pv, 5),
            'local_mean':          round(local_m, 4),
            'national_mean':       round(nat_m, 4),
            'deviation':           round(delta, 4),
            'contribution_to_risk': f"{'+' if contrib >= 0 else ''}{contrib}%",
        })
    return drivers[:10]

def compute_equity(raw_sub):
    n = len(raw_sub)
    if n == 0: return {}
    lit_map = {'Illiterate': 0, 'Semi-literate': 40, 'Literate': 100}
    inc_map = {'Low': 0, 'Lower-Middle': 25, 'Middle': 55, 'Upper-Middle': 75, 'High': 100}
    loc_map = {'Village': 20, 'Town': 60, 'City': 100}
    lit_s = sf(raw_sub['literacy_status'].map(lit_map).mean())
    inc_s = sf(raw_sub['income_bracket'].map(inc_map).mean())
    loc_s = sf(raw_sub['locality_type'].map(loc_map).mean())
    eld_r = sf((raw_sub['age'] >= 60).mean())
    eld_s = max(0, 100 - eld_r * 200)
    vax_s = sf(raw_sub['vaccination_up_to_date'].mean() * 100)
    eq    = int(max(0, min(100, lit_s*0.25 + inc_s*0.30 + loc_s*0.20 + eld_s*0.15 + vax_s*0.10)))
    vuln  = 'Critical' if eq < 40 else ('High' if eq < 55 else ('Medium' if eq < 70 else 'Low'))
    keys  = []
    if lit_s < 55:  keys.append('low_literacy')
    if inc_s < 40:  keys.append('low_income_households')
    if loc_s < 50:  keys.append('rural_population')
    if eld_r > 0.15: keys.append('high_elderly_ratio')
    if vax_s < 60:  keys.append('low_vaccination_coverage')
    if 'gender' in raw_sub.columns:
        if (raw_sub['gender'].str.lower() == 'female').mean() > 0.5:
            keys.append('female_headed_households')
    # equity-weighted risk
    scaled_sub = df_scaled.loc[raw_sub.index, FEATURES].fillna(df_scaled[FEATURES].mean())
    base_risk  = sf(BEST_MODEL.predict_proba(scaled_sub)[:, 1].mean()) * 100
    penalty    = (100 - eq) * 0.25
    eq_risk    = round(min(99.9, base_risk + penalty), 1)
    return {
        'equity_score':          eq,
        'vulnerability_level':   vuln,
        'literacy_score':        round(lit_s, 1),
        'income_score':          round(inc_s, 1),
        'locality_score':        round(loc_s, 1),
        'vaccination_score':     round(vax_s, 1),
        'elderly_ratio_pct':     round(eld_r * 100, 1),
        'key_factors':           keys if keys else ['no_critical_factors'],
        'equity_weighted_risk':  eq_risk,
    }

def compute_health_metrics(raw_sub):
    n = len(raw_sub)
    if n == 0: return {}
    scr_rate   = sf(raw_sub['screening_completed'].mean())
    exp_month  = max(1, int(n * 1.5))   # realistic monthly expectation
    act_month  = int(exp_month * scr_rate)
    gap_pct    = round((1 - scr_rate) * 100, 1)
    # workers_per_1000: inverse proxy from facility_access_km
    local_acc  = sf(raw_sub['facility_access_km'].mean())
    nat_acc    = sf(df['facility_access_km'].mean())
    w_local    = round(max(0.3, 4.0 - local_acc * 0.26), 2)
    w_state    = round(max(0.3, 4.0 - nat_acc  * 0.26), 2)
    # trend: compare halves
    mid        = n // 2
    if mid > 0:
        early = df.loc[raw_sub.index[:mid], 'ill'].mean()
        late  = df.loc[raw_sub.index[mid:], 'ill'].mean()
        chg   = round((late - early) / max(early, 0.001) * 100, 1)
        trend = f"Increasing (+{chg}% in last period)" if chg > 0 else f"Decreasing ({chg}% in last period)"
    else:
        trend = "Stable"
    return {
        'expected_tests_per_month':        exp_month,
        'actual_tests_conducted':          act_month,
        'gap_percent':                     gap_pct,
        'screening_completion_rate':       round(scr_rate * 100, 1),
        'healthcare_workers_per_1000':     w_local,
        'state_average_workers_per_1000':  w_state,
        'missed_screenings_avg':           round(sf(raw_sub['missed_screenings_12m'].mean()), 2),
        'missed_visits_avg':               round(sf(raw_sub['missed_visits_12m'].mean()), 2),
        'followup_delay_days_avg':         round(sf(raw_sub['followup_delay_days'].mean()), 1),
        'vaccination_coverage_pct':        round(sf(raw_sub['vaccination_up_to_date'].mean()) * 100, 1),
        'chronic_disease_burden_pct':      round(sf(raw_sub['chronic_disease'].mean()) * 100, 1),
        'current_trend':                   trend,
    }

def compute_weather():
    m = datetime.date.today().month
    cond, mult, warn = WEATHER_BY_MONTH.get(m, ("Unknown", 1.0, "No data"))
    return {
        'current_condition':           cond,
        'impact_multiplier':           mult,
        'effectiveness_reduction_pct': round((1 - mult) * 100, 1),
        'warning':                     warn,
    }

def compute_simulation(risk_pct):
    interventions = [
        {'type': 'mobile_clinics', 'count': 4, 'impact': '+18% testing capacity', '_wr': 0.9},
        {'type': 'asha_groups',    'count': 6, 'impact': '+9% followup compliance', '_wr': 0.5},
    ]
    total_wr = sum(i['_wr'] for i in interventions)
    weekly   = [round(risk_pct, 1)]
    cur      = risk_pct
    for _ in range(11):
        red = total_wr * (cur / 100) * 0.6
        cur = max(28.0, cur - red)
        weekly.append(round(cur, 1))
    total_red = round(risk_pct - weekly[-1], 1)
    clean_iv  = [{'type': i['type'], 'count': i['count'], 'impact': i['impact']}
                  for i in interventions]
    return {
        'active':                   True,
        'applied_interventions':    clean_iv,
        'projected_risk_reduction': total_red,
        'weeks_simulated':          12,
        'projected_weekly_risk':    weekly,
    }

def compute_optimization(risk_pct, equity_score, drivers):
    sev = risk_pct / 100
    mc  = max(2, int(sev * 8))
    ag  = max(3, int(sev * 12) + (2 if equity_score < 60 else 0))
    sc  = max(1, int(sev * 5))
    rr  = round(15 + sev * 12, 1)
    eg  = max(5, 25 - equity_score // 5)
    cp  = int(1200 + (1 - sev) * 1000)
    top = drivers[0]['label'] if drivers else 'Missed Screenings'
    return {
        'recommended_allocation': {
            'mobile_clinics':  mc,
            'asha_groups':     ag,
            'screening_camps': sc,
        },
        'priority_target': top,
        'expected_outcome': {
            'risk_reduction':  rr,
            'equity_gain':     eg,
            'cost_efficiency': f'₹{cp:,} per additional screening',
        },
    }

def compute_counterfactuals(risk_pct, drivers):
    SCENARIOS = {
        'missed_screenings_12m':  ('If missed screenings reduced by 50%',      0.12),
        'missed_visits_12m':      ('If missed visits reduced by 40%',           0.09),
        'followup_delay_days':    ('If followup delay days reduced by 40 days', 0.11),
        'facility_access_km':     ('If a healthcare facility added within 5 km',0.08),
        'vaccination_up_to_date': ('If vaccination coverage raised to 90%',     0.07),
        'obesity':                ('If obesity rate reduced by 20%',            0.06),
        'region_health_index':    ('If regional health index improved by 0.5 SD',0.05),
        'chronic_disease':        ('If chronic disease management improved 30%', 0.04),
    }
    out = []
    for d in drivers[:6]:
        feat = d['feature']
        if feat in SCENARIOS:
            txt, factor = SCENARIOS[feat]
            red = round(risk_pct * factor, 1)
            out.append({'scenario':       txt,
                        'driver':         d['label'],
                        'risk_reduction': red,
                        'new_risk_pct':   round(max(10.0, risk_pct - red), 1)})
    return out

def compute_copilot(location, risk_pct, risk_level, drivers, equity, health, weather):
    td  = drivers[0]['label'] if drivers else 'unknown'
    td2 = drivers[1]['label'] if len(drivers) > 1 else ''
    ev  = equity.get('vulnerability_level', '')
    gap = health.get('gap_percent', 0)
    ww  = weather.get('warning', '')
    wm  = weather.get('impact_multiplier', 1.0)
    parts = [f"{location} is at {risk_level.lower()} illness risk ({risk_pct}%)."]
    parts.append(f"Primary driver: '{td}'" + (f"; also '{td2}'." if td2 else '.'))
    if gap > 50:
        parts.append(f"Testing is critically under-delivered — {gap}% gap vs expected.")
    if ev in ('High', 'Critical'):
        kf = ', '.join(equity.get('key_factors', [])[:3]).replace('_', ' ')
        parts.append(f"Equity vulnerability is {ev} due to: {kf}.")
    if wm < 0.85:
        parts.append(f"Weather reduces intervention effectiveness by {round((1-wm)*100)}%. {ww}")
    parts.append("Prioritize ASHA deployment in high-vulnerability pockets for the best risk-equity tradeoff.")
    return ' '.join(parts)

print("✅ All helper functions defined")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# PRECOMPUTE FULL JSON FOR EVERY STATE AND DISTRICT
# ═══════════════════════════════════════════════════════════════
cache = {'states': {}, 'districts': {}}
weather = compute_weather()

all_states    = sorted(df['state_or_ut'].unique())
total_states  = len(all_states)
total_distrs  = 0

print(f"Computing predictions for {total_states} states + all their districts...")
print("="*70)

for s_idx, state_name in enumerate(all_states):
    raw_state    = df[df['state_or_ut'] == state_name]
    scaled_state = df_scaled[df_scaled['state_or_ut'] == state_name]

    # ── State-level metrics ────────────────────────────────────
    ml      = compute_ml_risk(scaled_state)
    drivers = compute_drivers(scaled_state)
    equity  = compute_equity(raw_state)
    health  = compute_health_metrics(raw_state)
    sim     = compute_simulation(ml['current_risk_pct'])
    opt     = compute_optimization(ml['current_risk_pct'], equity['equity_score'], drivers)
    cf      = compute_counterfactuals(ml['current_risk_pct'], drivers)
    copilot = compute_copilot(state_name, ml['current_risk_pct'], ml['risk_level'],
                               drivers, equity, health, weather)

    cache['states'][state_name.lower()] = {
        '_display_name':            state_name,
        'ml_risk_assessment':       ml,
        'top_risk_drivers':         drivers,
        'equity_vulnerability':     equity,
        'health_metrics':           health,
        'weather_context':          weather,
        'simulation_state':         sim,
        'optimization_suggestions': opt,
        'counterfactuals':          cf,
        'explanation_for_ai_copilot': copilot,
        'districts':                sorted(raw_state['district_name'].unique().tolist()),
        'sample_size':              len(raw_state),
    }

    # ── District-level metrics ─────────────────────────────────
    cache['districts'][state_name.lower()] = {}
    for dist_name in sorted(raw_state['district_name'].unique()):
        raw_dist    = raw_state[raw_state['district_name'] == dist_name]
        scaled_dist = df_scaled.loc[raw_dist.index]

        ml_d  = compute_ml_risk(scaled_dist)
        drv_d = compute_drivers(scaled_dist)
        eq_d  = compute_equity(raw_dist)
        hm_d  = compute_health_metrics(raw_dist)
        sim_d = compute_simulation(ml_d['current_risk_pct'])
        opt_d = compute_optimization(ml_d['current_risk_pct'], eq_d['equity_score'], drv_d)
        cf_d  = compute_counterfactuals(ml_d['current_risk_pct'], drv_d)
        cop_d = compute_copilot(f"{dist_name}, {state_name}", ml_d['current_risk_pct'],
                                 ml_d['risk_level'], drv_d, eq_d, hm_d, weather)

        cache['districts'][state_name.lower()][dist_name.lower()] = {
            '_display_name':            dist_name,
            '_state_display':           state_name,
            'ml_risk_assessment':       ml_d,
            'top_risk_drivers':         drv_d,
            'equity_vulnerability':     eq_d,
            'health_metrics':           hm_d,
            'weather_context':          weather,
            'simulation_state':         sim_d,
            'optimization_suggestions': opt_d,
            'counterfactuals':          cf_d,
            'explanation_for_ai_copilot': cop_d,
            'sample_size':              len(raw_dist),
        }
        total_distrs += 1

    print(f"  [{s_idx+1:02d}/{total_states}] {state_name:<40} risk={cache['states'][state_name.lower()]['ml_risk_assessment']['current_risk_pct']}%  districts={len(raw_state['district_name'].unique())}")

print(f"\n✅ Precomputed {total_states} states + {total_distrs} districts")


In [ ]:
# ── Save prediction cache ──────────────────────────────────────────────────
with open('preventiq_cache.json', 'w') as f:
    json.dump(cache, f, indent=2)
print(f"✅ preventiq_cache.json saved ({Path('preventiq_cache.json').stat().st_size / 1024:.0f} KB)")

# ── Save model metadata ────────────────────────────────────────────────────
meta = {
    'model_name':       BEST_NAME,
    'accuracy':         BEST_META['Accuracy'],
    'f1_score':         BEST_META['F1'],
    'roc_auc':          BEST_META['ROC-AUC'],
    'features':         FEATURES,
    'scaled_cols':      COLS_TO_SCALE,
    'cat_features':     CAT_FEATURES,
    'feature_importances': {k: round(v, 6) for k,v in importances.items()},
    'training_date':    datetime.date.today().isoformat(),
    'training_samples': int(X_train.shape[0]),
    'n_features':       len(FEATURES),
}
with open(MODEL_DIR / 'model_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
print(f"✅ models/model_meta.json saved")

print("\n" + "="*60)
print("FILES TO SEND TO YOUR DEVELOPER:")
print("="*60)
print("  preventiq_cache.json      ← precomputed JSON for all locations")
print("  models/best_model.joblib  ← trained ML model")
print("  models/scaler.joblib      ← feature scaler")
print("  models/model_meta.json    ← model metadata & features")
print("  final_health_data.xlsx    ← original dataset")
print("  preventiq_backend.py      ← FastAPI backend (already given)")


In [ ]:
# ── Model performance chart ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('PreventIQ — Model Training Summary', fontsize=14, fontweight='bold')

# F1 scores
ax = axes[0]
colors = ['#2ecc71' if i==0 else '#3498db' for i in range(len(results_df))]
bars = ax.barh(results_df['Model'], results_df['F1'], color=colors)
ax.set_title('F1-Score Comparison', fontweight='bold'); ax.set_xlim([0, 1.05])
for b, v in zip(bars, results_df['F1']):
    ax.text(v+0.01, b.get_y()+b.get_height()/2, f'{v:.4f}', va='center', fontsize=9)

# Confusion matrix
ax = axes[1]
yp = BEST_MODEL.predict(X_test)
cm = confusion_matrix(y_test, yp)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Not Ill','Ill'], yticklabels=['Not Ill','Ill'])
ax.set_title(f'Confusion Matrix\n({BEST_NAME})', fontweight='bold')

# Feature importance
ax = axes[2]
fi = pd.Series({FRIENDLY_NAMES.get(k, k): v for k,v in importances.items()}).sort_values()
fi.tail(10).plot(kind='barh', ax=ax, color='#e74c3c')
ax.set_title('Top 10 Feature Importances', fontweight='bold')

plt.tight_layout()
plt.savefig('training_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ training_summary.png saved")


In [ ]:
# ── Quick sanity check: peek at Indore output ──────────────────────────────
indore = cache['districts'].get('madhya pradesh', {}).get('indore', None)
if indore:
    print("\n=== SAMPLE OUTPUT: Indore, Madhya Pradesh ===")
    print(json.dumps({
        'ml_risk_assessment':       indore['ml_risk_assessment'],
        'equity_vulnerability':     {k: indore['equity_vulnerability'][k]
                                     for k in ['equity_score','vulnerability_level','key_factors','equity_weighted_risk']},
        'top_risk_drivers':         indore['top_risk_drivers'][:3],
        'counterfactuals':          indore['counterfactuals'][:2],
        'explanation_for_ai_copilot': indore['explanation_for_ai_copilot'],
    }, indent=2))
else:
    print("Indore not found — check district name in dataset")
